# Zero-shot soccer next-event prediction with Qwen3-1.7B

Baselines for mechanistic-interpretability work: predict the next StatsBomb event (one of 15 classes) from a 16-event match context, reading the model's first-token logits over single-letter class codes (`a`–`o`).

Three feature-set runs — **identity+context** (all 18 features), **identity-only** (4: player, team, role, competition), **context-only** (14: positions, times, scores, previous-action features) — each reported under two read-outs: **A**, softmax over the full vocabulary restricted to the 15 label tokens, and **B**, softmax renormalised over the 15 tokens (headline; comparable to the trained Track-A baseline, NLL ≈ 0.89–0.93). Metrics: NLL, Accuracy, Macro-F1, ECE, plus per-class reports and prediction distributions.

Run top-to-bottom on a Colab T4 GPU. All event data comes from `events_prepared.parquet`; the only external files are id→name dictionaries for readable team/competition names. Per-run probabilities are saved as `.npz` artefacts on Drive, and two audit cells verify the inference loop.

In [ ]:
import sys, platform, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3-1.7B"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda")
model.eval()

msgs = [{"role": "user", "content": "Give one fun fact about soccer in exactly one sentence."}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

inputs = tok([text], return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

print("PROMPT:\n", text)
print("OUTPUT:\n", tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

PROMPT:
 <|im_start|>user
Give one fun fact about soccer in exactly one sentence.<|im_end|>
<|im_start|>assistant

OUTPUT:
 <think>
Okay, the user wants a fun fact about soccer in one sentence. Let me think. Soccer is the most popular sport in the world, right? But I need something specific. Oh, did you know that the word "soccer" comes from the English word "football"? That's a fun fact. But wait, is that accurate? I think the original term was "football,"


## Next-Event Prediction with Qwen3-1.7B — Feature-Set Baselines

**Goal:** 16-event context → predict next event, one of 15 `action` classes from `events_prepared.parquet`.

Each event class is mapped to a single letter token (`a`–`o`). The model's pre-softmax logits at the next-token position are read out for exactly those 15 token IDs → probability distribution over events → NLL / Accuracy / Macro-F1 / ECE, under two read-outs:

- **A (full vocab):** softmax over the full vocabulary, then take the 15 label tokens
- **B (renormalised):** softmax over the 15 label tokens only — headline, comparable to the Track A baseline (NLL ≈ 0.89–0.93)

**Three baseline runs (for later mech-interp work):**
1. **identity+context** — all 18 features (4 identity + 14 context), zero-shot, random sample of the test set
2. **identity** — 4 identity features only (`player, team, role, competition`), full test set
3. **context** — 14 context features only, full test set

Notebook flow: (1) model + label/prompt verification, (2) data prep → 16-event windows, (3) inference engine (fp16 + memory-efficient attention), (4) identity+context baseline, (5) shared pipeline + identity/context runs, (6) audit diagnostics, (7) artefacts + results summary. All tunables (`CONTEXT_LEN`, `STRIDE`, `SPLIT`, `FEATURE_SETS`, …) are variables — reuse for new feature subsets.

In [ ]:
# ============================================================
# Label mapping: 15 events -> letters a..o
# Verify each letter is EXACTLY ONE token in Qwen3's tokenizer.
# ============================================================
LABELS  = list("abcdefghijklmno")
EVENTS  = ["PASS_SHORT","PASS_LONG","CARRY","CROSS","CLEARANCE","RECOVERY",
           "DUEL","INTERCEPTION","SHOT","GK_ACTION","DRIBBLE","LOSS",
           "FOUL_COMMITTED","FOUL_WON","OTHER"]
LABEL2EVENT = dict(zip(LABELS, EVENTS))
EVENT2LABEL = {e: l for l, e in LABEL2EVENT.items()}

print(f"{'label':<6} {'token_id':>9} {'token_string':<14} {'decodes_to':<8} event")
print("-" * 70)
for l in LABELS:
    ids = tok.encode(l, add_special_tokens=False)
    assert len(ids) == 1, f"label {l!r} tokenized to {len(ids)} tokens: {ids}"
    tid = ids[0]
    ts  = tok.convert_ids_to_tokens([tid])[0]
    dec = tok.decode([tid])
    assert dec == l, f"label {l!r} decodes to {dec!r}"
    print(f"{l:<6} {tid:>9} {ts:<14} {dec!r:<8} {LABEL2EVENT[l]}")

# vocab-wide sanity: are ALL single ascii letters one token?
for ch in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789":
    assert len(tok.encode(ch, add_special_tokens=False)) == 1, ch
print("\nOK: every single ASCII letter/digit is exactly 1 token; a..o mapping verified.")

label   token_id token_string   decodes_to event
----------------------------------------------------------------------
a             64 a              'a'      PASS_SHORT
b             65 b              'b'      PASS_LONG
c             66 c              'c'      CARRY
d             67 d              'd'      CROSS
e             68 e              'e'      CLEARANCE
f             69 f              'f'      RECOVERY
g             70 g              'g'      DUEL
h             71 h              'h'      INTERCEPTION
i             72 i              'i'      SHOT
j             73 j              'j'      GK_ACTION
k             74 k              'k'      DRIBBLE
l             75 l              'l'      LOSS
m             76 m              'm'      FOUL_COMMITTED
n             77 n              'n'      FOUL_WON
o             78 o              'o'      OTHER

OK: every single ASCII letter/digit is exactly 1 token; a..o mapping verified.


In [ ]:
# ============================================================
# Prompt construction.
# Requirements: model output must be STRICTLY ONE LETTER (a..o),
# i.e. the letter must be exactly the NEXT token after the prompt,
# and nothing else may be valid. Verify with the tokenizer (no model).
# ============================================================

def build_prompt(context_text: str) -> str:
    """context_text: rendered 16-event context (feature formatting added in later cells).
    Returns the full prompt string (chat template already applied)."""
    legend = ", ".join(f"{l}={e}" for l, e in LABEL2EVENT.items())
    user = (
        "You are predicting the next event in a soccer match.\n"
        f"Event codes: {legend}\n"
        f"Context (past 16 events, oldest -> newest):\n{context_text}\n"
        "Reply with EXACTLY ONE letter from a-o and nothing else."
    )
    msgs = [{"role": "user", "content": user}]
    # enable_thinking=False: no <think> block, answer starts immediately
    return tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

# --- verification: letter must be exactly the single next token ---
dummy_context = "\n".join(f"1. {e}" for e in EVENTS)  # placeholder context
p = build_prompt(dummy_context)
base_ids = tok.encode(p, add_special_tokens=False)

print("prompt tail:", repr(p[-60:]))
print(f"\n{'label':<6} {'next_token_id':>13} {'token_string':<14} n_diff_tokens")
print("-" * 55)
for l in LABELS:
    ids = tok.encode(p + l, add_special_tokens=False)
    diff = ids[len(base_ids):]
    assert len(diff) == 1, f"label {l!r} is {len(diff)} tokens in prompt context: {diff}"
    ts = tok.convert_ids_to_tokens([diff[0]])[0]
    print(f"{l:<6} {diff[0]:>13} {ts:<14} {len(diff)}")

print("\nOK: after the prompt, each label letter is EXACTLY ONE token ->")
print("    next-token logits at that position can be sliced to these 15 token IDs.")

prompt tail: 'ng else.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

label  next_token_id token_string   n_diff_tokens
-------------------------------------------------------
a                 64 a              1
b                 65 b              1
c                 66 c              1
d                 67 d              1
e                 68 e              1
f                 69 f              1
g                 70 g              1
h                 71 h              1
i                 72 i              1
j                 73 j              1
k                 74 k              1
l                 75 l              1
m                 76 m              1
n                 77 n              1
o                 78 o              1

OK: after the prompt, each label letter is EXACTLY ONE token ->
    next-token logits at that position can be sliced to these 15 token IDs.


## Baseline 1 — identity + context (all 18 features)

Mirrors the full Track A model (`ALL_GROUPS` in `src/model.py`): every past event is rendered with all **18** features the trained baseline consumed — 4 identity + 14 context. Excludes the dead `play_pattern` and the unused parquet extras (`second`, `t_period_s`, `end_zone`, `coupled_with_prev`), exactly like the notebooks.

The cells below load `events_prepared.parquet`, build the (16-event context → next event) windows for the test split, and render prompts — setup shared by all three baseline runs.

Requires the cells above (model, labels, `build_prompt`) to have been run first.

In [ ]:
# ============================================================
# Run 3 data prep: load parquet, build (16-event context -> next event)
# windows, render each past event with its 18 features
# (4 identity + 14 context) exactly as the Track A full model consumed them.
# Pre-action sourcing: poss_len<-prev_poss_len, possession_flag<-team_change,
# t_poss<-t_poss_prev (same as POSSESSION_SOURCE_COLUMNS['pre_action']).
# ============================================================
import pandas as pd

PARQUET = "/content/drive/MyDrive/soccer_mi/model_training_evaluation/artifacts/events_prepared.parquet"

CONTEXT_LEN = 16
STRIDE = 8            # windows every 8 events, as in Track A
SPLIT = "test"        # evaluate on the held-out test split

df = pd.read_parquet(PARQUET)
print("loaded:", df.shape)

# prev_action_id -> readable name.
# Verified in the cell above: action_id runs 0-14 (15 classes), and
# prev_action_id == 15 is the "no previous event" sentinel (vocab size 16).
action_map = df.dropna(subset=["action"]).groupby("action_id")["action"].first().to_dict()
prev_action_map = {**action_map, 15: "END"}

# --- render one event with its 18 features (rows are plain dicts) ---
def render_event(r) -> str:
    return (
        f"player={r['player_name']}; team={r['team_id']}; role={r['position_name']}; "
        f"competition={r['competition_id']}; zone={int(r['zone'])}; "
        f"x={r['x_norm']:.3f}; y={r['y_norm']:.3f}; period={int(r['period'])}; "
        f"minute_bin={int(r['minute_bin'])}; dt={r['dt']:.1f}; "
        f"t_poss={r['t_poss_prev']:.1f}; score_diff={int(r['score_diff_idx'])}; "
        f"poss_len={int(r['prev_poss_len'])}; possession_flag={int(r['team_change'])}; "
        f"prev_action={prev_action_map.get(int(r['prev_action_id']), r['prev_action_id'])}; "
        f"prev_outcome={int(r['prev_outcome'])}; prev_end_zone={int(r['prev_end_zone'])}; "
        f"same_prev_player={int(r['same_player_as_prev'])}"
    )

def build_windows(frame, context_len=CONTEXT_LEN, stride=STRIDE):
    """(context_rows, target_row) pairs within the same match, never crossing matches."""
    wins = []
    for _, grp in frame.groupby("match_id", sort=False):
        rows = grp.to_dict("records")
        for i in range(context_len, len(rows)):
            wins.append((rows[i - context_len : i], rows[i]))
    wins = wins[::stride]
    return wins

test_df = df[df["split"] == SPLIT].reset_index(drop=True)
windows = build_windows(test_df)
print(f"split={SPLIT}, context={CONTEXT_LEN}, stride={STRIDE} -> {len(windows)} windows")

# --- sample render, no model involved ---
ctx_rows, target = windows[0]
print("\n--- sample past event (oldest) ---\n", render_event(ctx_rows[0]))
print("\n--- sample past event (newest) ---\n", render_event(ctx_rows[-1]))
print("\n--- target ---\n", target["action"], "-> label", EVENT2LABEL[target["action"]])

loaded: (658905, 46)
split=test, context=16, stride=8 -> 8057 windows

--- sample past event (oldest) ---
 player=Luis Alberto Suárez Díaz; team=217; role=Center Forward; competition=11; zone=52; x=0.500; y=0.500; period=1; minute_bin=0; dt=0.0; t_poss=0.0; score_diff=3; poss_len=21; possession_flag=0; prev_action=END; prev_outcome=2; prev_end_zone=52; same_prev_player=0

--- sample past event (newest) ---
 player=Jesús Navas González; team=213; role=Right Midfield; competition=11; zone=45; x=0.433; y=0.700; period=1; minute_bin=0; dt=0.9; t_poss=0.9; score_diff=3; poss_len=0; possession_flag=0; prev_action=PASS_SHORT; prev_outcome=0; prev_end_zone=45; same_prev_player=0

--- target ---
 PASS_SHORT -> label a


In [ ]:
# ============================================================
# Render every window into a full prompt string (all features in context).
# Also re-verify: with REAL context text, each label letter is still exactly
# the single next token (context can change BPE boundaries -> assert again).
# ============================================================
from tqdm.auto import tqdm

# label letter -> its (single) token id, captured once
LABEL_IDS = {l: tok.encode(l, add_special_tokens=False)[0] for l in LABELS}

def render_context(ctx_rows) -> str:
    lines = [f"{k+1}. {render_event(r)}" for k, r in enumerate(ctx_rows)]
    return "\n".join(lines)

def prompt_for(window) -> str:
    ctx_rows, target = window
    return build_prompt(render_context(ctx_rows)), EVENT2LABEL[target["action"]]

# verify on 200 random windows that the label letter stays a single next token
import random
random.seed(0)
sample = random.sample(windows, min(200, len(windows)))
for w in sample:
    p, lab = prompt_for(w)
    base = tok.encode(p, add_special_tokens=False)
    ids = tok.encode(p + lab, add_special_tokens=False)
    diff = ids[len(base):]
    assert len(diff) == 1 and diff[0] == LABEL_IDS[lab], f"diff={diff} for label {lab}"
print(f"OK: {len(sample)}/{len(sample)} sampled prompts keep every label as exactly 1 token")

# materialise all prompts (strings; batching/tokenising happens in the inference cell)
prompts, labels = zip(*[prompt_for(w) for w in tqdm(windows, desc="rendering prompts")])
print(f"rendered {len(prompts)} prompts; e.g.\n\n{prompts[0][:600]}...")

OK: 200/200 sampled prompts keep every label as exactly 1 token


rendering prompts:   0%|          | 0/8057 [00:00<?, ?it/s]

rendered 8057 prompts; e.g.

<|im_start|>user
You are predicting the next event in a soccer match.
Event codes: a=PASS_SHORT, b=PASS_LONG, c=CARRY, d=CROSS, e=CLEARANCE, f=RECOVERY, g=DUEL, h=INTERCEPTION, i=SHOT, j=GK_ACTION, k=DRIBBLE, l=LOSS, m=FOUL_COMMITTED, n=FOUL_WON, o=OTHER
Context (past 16 events, oldest -> newest):
1. player=Luis Alberto Suárez Díaz; team=217; role=Center Forward; competition=11; zone=52; x=0.500; y=0.500; period=1; minute_bin=0; dt=0.0; t_poss=0.0; score_diff=3; poss_len=21; possession_flag=0; prev_action=END; prev_outcome=2; prev_end_zone=52; same_prev_player=0
2. player=Sergio Busquets i Bur...


In [ ]:
# ============================================================
# Run 3 - inference setup.
# NOTE: all event/target data comes from events_prepared.parquet.
# The only external files here are the id->name dictionaries used to
# render readable team/competition names in prompts (hudl/open-data
# via jsdelivr; matches files carry home/away team sub-dicts whose keys
# are home_team_id / home_team_name etc).
# TUNABLES (change these, everything downstream adapts):
#   TARGET_MINUTES : budget for this inference pass (auto-sizes N_SAMPLES
#                    via a measured per-batch time; None = use all windows)
#   BATCH_SIZE     : prompts per forward pass
#   N_SAMPLES      : explicit window count; overrides TARGET_MINUTES if set
# ============================================================
import requests, json, math, time

TARGET_MINUTES = 20
BATCH_SIZE     = 8
N_SAMPLES      = None   # None -> derive from TARGET_MINUTES

CDN = "https://cdn.jsdelivr.net/gh/hudl/open-data@master/data"

def fetch_json(path):
    r = requests.get(f"{CDN}/{path}", timeout=30)
    if not r.ok:
        print(f"  miss {path} ({r.status_code})")
        return None
    return r.json()

comps = fetch_json("competitions.json")
COMP_NAME = {c["competition_id"]: c["competition_name"] for c in comps}

# team names live inside matches files: m['home_team'] = {home_team_id, home_team_name, ...}
needed = set(int(c) for c in df["competition_id"].dropna().unique())
TEAM_NAME = {}
for cid in sorted(needed):
    seasons = [c["season_id"] for c in comps if c["competition_id"] == cid]
    for sid in seasons:
        matches = fetch_json(f"matches/{cid}/{sid}.json")
        if matches is None:
            continue
        for m in matches:
            for side in ("home", "away"):
                t = m.get(f"{side}_team") or {}
                tid, tnm = t.get(f"{side}_team_id"), t.get(f"{side}_team_name")
                if tid is not None and tnm is not None:
                    TEAM_NAME[tid] = tnm

missing_team_ids = sorted(set(int(t) for t in df["team_id"].dropna().unique()) - set(TEAM_NAME))
print(f"loaded {len(TEAM_NAME)} team names, {len(COMP_NAME)} competition names")
print("team ids without a name (render as raw id):", missing_team_ids or "none")

# --- re-render events with names (overrides the id-only version above) ---
def render_event(r) -> str:
    return (
        f"player={r['player_name']}; team={TEAM_NAME.get(int(r['team_id']), r['team_id'])}; "
        f"role={r['position_name']}; "
        f"competition={COMP_NAME.get(int(r['competition_id']), r['competition_id'])}; "
        f"zone={int(r['zone'])}; "
        f"x={r['x_norm']:.3f}; y={r['y_norm']:.3f}; period={int(r['period'])}; "
        f"minute_bin={int(r['minute_bin'])}; dt={r['dt']:.1f}; "
        f"t_poss={r['t_poss_prev']:.1f}; score_diff={int(r['score_diff_idx'])}; "
        f"poss_len={int(r['prev_poss_len'])}; possession_flag={int(r['team_change'])}; "
        f"prev_action={prev_action_map.get(int(r['prev_action_id']), r['prev_action_id'])}; "
        f"prev_outcome={int(r['prev_outcome'])}; prev_end_zone={int(r['prev_end_zone'])}; "
        f"same_prev_player={int(r['same_player_as_prev'])}"
    )

# re-materialise prompts with names
prompts, labels = zip(*[prompt_for(w) for w in tqdm(windows, desc="re-rendering with names")])
print(f"\nrendered {len(prompts)} prompts with readable team/competition names")
print("\nexample event line:\n", render_event(windows[0][0][-1]))

loaded 350 team names, 24 competition names
team ids without a name (render as raw id): none


re-rendering with names:   0%|          | 0/8057 [00:00<?, ?it/s]


rendered 8057 prompts with readable team/competition names

example event line:
 player=Jesús Navas González; team=Sevilla; role=Right Midfield; competition=La Liga; zone=45; x=0.433; y=0.700; period=1; minute_bin=0; dt=0.9; t_poss=0.9; score_diff=3; poss_len=0; possession_flag=0; prev_action=PASS_SHORT; prev_outcome=0; prev_end_zone=45; same_prev_player=0


## Inference engine

`run_batch` (defined in the timing cell below) is the single forward pass every run in this notebook uses: fp16 weights with PyTorch's memory-efficient attention kernel, `logits_to_keep=1` (last position only), no KV cache, logits cast to fp32. fp16 is required on this hardware — with bf16, PyTorch falls back to slow math attention on a T4 (sm_75; FlashAttention needs sm_80+), which OOMs above batch 1. The timing block calibrates per-batch speed and derives sample sizes from it.

In [ ]:
# ============================================================
# Run 3: timing calibration.
# Key findings (backend-test cell):
#   - FlashAttention needs sm80+ (T4 is sm7.5) -> never available here
#   - PyTorch's memory-efficient SDPA kernel accepts {fp16, fp32}, NOT bf16
#     -> with bf16 everything fell back to math attention (6.5 GiB at 7.3k
#        tokens: OOM at batch>1, ~17 s/prompt even at batch 1)
#   So: model runs in fp16 and forward passes force the efficient kernel
#   (math kept as fallback). Measured 0.12 s attention for batch 2 at 7365.
#   logits_to_keep=1 -> only last position; use_cache=False -> no KV cache.
# ============================================================
import numpy as np
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

model = model.to(torch.float16)          # fp16: unlocks memory-efficient SDPA
print(f"model dtype: {model.dtype}")

label_id_list = [LABEL_IDS[l] for l in LABELS]        # 15 token ids
label_id_tensor = torch.tensor(label_id_list, device=model.device)

# --- timing calibration on a small batch ---
def run_batch(prompt_batch):
    enc = tok(prompt_batch, return_tensors="pt", padding=True,
              padding_side="left", add_special_tokens=False).to(model.device)
    with torch.no_grad(), sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):
        # logits_to_keep=1: only the final position's logits -> [B, 1, vocab]
        # use_cache=False: no KV cache allocated (single forward pass)
        out = model(**enc, logits_to_keep=1, use_cache=False)
    logits = out.logits[:, 0].float()                     # [B, vocab] fp32
    return logits

torch.cuda.empty_cache()
t0 = time.time()
_ = run_batch(prompts[:BATCH_SIZE])                          # warm-up (excludes first-call overhead)
t0 = time.time()
_ = run_batch(prompts[:BATCH_SIZE])
per_batch_s = (time.time() - t0) / 1

budget_s = TARGET_MINUTES * 60 if TARGET_MINUTES else float("inf")
if N_SAMPLES is None:
    n_samples = min(len(prompts), int(budget_s / per_batch_s) * BATCH_SIZE)
else:
    n_samples = min(len(prompts), N_SAMPLES)
n_batches = math.ceil(n_samples / BATCH_SIZE)
est_s = n_batches * per_batch_s

print(f"measured: {per_batch_s:.2f} s/batch of {BATCH_SIZE}")
print(f"budget:   {TARGET_MINUTES} min | n_samples={n_samples} "
      f"({n_samples/len(prompts)*100:.1f}% of {len(prompts)}) | "
      f"batches={n_batches} | est {est_s/60:.1f} min")

model dtype: torch.float16
measured: 1.54 s/batch of 8
budget:   20 min | n_samples=6224 (77.2% of 8057) | batches=778 | est 20.0 min


## Baseline 1 run — zero-shot prompt + random sampling

Final baseline config (the `V2` dict below is the single place for these knobs):

1. **Zero-shot** (`n_shot=0`) — a 3-shot variant was tested and *collapsed* predictions to 100% PASS_SHORT with worse macro-F1 than zero-shot (in-context anchoring), so the clean baseline is zero-shot
2. **Random sampling** of test windows (`random_sample`, `sample_seed`) instead of first-N (first-N was skewed toward early matches)
3. **Instruction wording** as a tunable (`final_instruction`)

**Result (2,448 random test windows, 30% of the test set):** B read-out NLL 3.39 | Acc 0.288 | Macro-F1 0.030 | ECE 0.572 — model predicts PASS_SHORT for **100%** of windows (majority rate 28.8%). A degenerate constant predictor: no discriminative use of the context. Saved to `run_identity_context.npz` (via the cell below).

In [ ]:
# ============================================================
# Run 3 v2: tunables -> few-shot prompt + random sampling.
# V2 dict is the single place to change these; reuse for later runs.
# NOTE (v3 config): n_shot=0 DISABLES few-shot. The 3-shot run collapsed
# predictions to 100% PASS_SHORT (macro-F1 0.033, worse than zero-shot),
# so the clean baseline is the v1 zero-shot prompt + random sampling.
# Setting n_shot>0 reinstates few-shot for future experiments.
# ============================================================
V2 = dict(
    n_shot          = 0,     # 0 = zero-shot (clean baseline); >0 = few-shot
    random_sample   = True,  # sample test windows randomly instead of first-N
    sample_seed     = 0,
    final_instruction = "Given the context, the next event code is:",  # keep ending in ":" so the letter stays a lone token
)

# --- few-shot exemplars from the TRAIN split (no test leakage) ---
exemplars = []
if V2["n_shot"] > 0:
    train_windows = build_windows(df[df["split"] == "train"].reset_index(drop=True))
    print(f"train windows available: {len(train_windows)}")
    train_lab = [EVENT2LABEL[w[1]["action"]] for w in train_windows]
    from collections import Counter
    common = [l for l, _ in Counter(train_lab).most_common()]
    for letter in common:                       # one per class, in frequency order
        for w in train_windows:
            if EVENT2LABEL[w[1]["action"]] == letter:
                exemplars.append(w)
                break
        if len(exemplars) == V2["n_shot"]:
            break
    print("exemplars:", [(EVENT2LABEL[w[1]["action"]], LABEL2EVENT[EVENT2LABEL[w[1]["action"]]]) for w in exemplars])
else:
    print("n_shot=0 -> zero-shot (no few-shot block)")

def build_prompt_v2(context_text, exemplars=exemplars):
    """Same structure as build_prompt, plus optional few-shot block."""
    legend = ", ".join(f"{l}={e}" for l, e in LABEL2EVENT.items())
    lines = ["You are predicting the next event in a soccer match.",
             f"Event codes: {legend}"]
    for k, (ctx_rows, tgt) in enumerate(exemplars, 1):
        lines.append(f"Example {k}:")
        lines.append(render_context(ctx_rows))
        lines.append(f"{V2['final_instruction']} {EVENT2LABEL[tgt['action']]}")
    lines.append("Context (past 16 events, oldest -> newest):")
    lines.append(context_text)
    lines.append(V2["final_instruction"])
    user = "\n".join(lines)
    msgs = [{"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

# --- random sampling of the test windows ---
test_windows = windows   # from the data-prep cell (test split, stride 8)
if V2["random_sample"]:
    rng = random.Random(V2["sample_seed"])
    test_windows = rng.sample(test_windows, len(test_windows))
print(f"eval order: {'random (seed '+str(V2['sample_seed'])+')' if V2['random_sample'] else 'sequential'}")

# --- render v2 prompts + re-verify single-token property ---
v2_prompts, v2_labels = zip(*[
    (build_prompt_v2(render_context(w[0])), EVENT2LABEL[w[1]["action"]])
    for w in tqdm(test_windows, desc="v2 prompts")
])

random.seed(1)
for i in random.sample(range(len(v2_prompts)), min(100, len(v2_prompts))):
    p, lab = v2_prompts[i], v2_labels[i]
    base = tok.encode(p, add_special_tokens=False)
    diff = tok.encode(p + lab, add_special_tokens=False)[len(base):]
    assert len(diff) == 1 and diff[0] == LABEL_IDS[lab], f"i={i} diff={diff}"
print(f"OK: 100/100 sampled v2 prompts keep the label as exactly 1 next token")
print(f"\nv2 prompt length (chars): {len(v2_prompts[0])}; e.g.\n\n{v2_prompts[0][-500:]}")

n_shot=0 -> zero-shot (no few-shot block)
eval order: random (seed 0)


v2 prompts:   0%|          | 0/8057 [00:00<?, ?it/s]

OK: 100/100 sampled v2 prompts keep the label as exactly 1 next token

v2 prompt length (chars): 5051; e.g.

f=5; poss_len=16; possession_flag=0; prev_action=RECOVERY; prev_outcome=2; prev_end_zone=67; same_prev_player=1
16. player=Linda Lizeth Caicedo Alegría; team=Real Madrid CF W; role=Left Midfield; competition=Liga F; zone=75; x=0.754; y=0.377; period=2; minute_bin=17; dt=1.5; t_poss=4.0; score_diff=5; poss_len=17; possession_flag=0; prev_action=CARRY; prev_outcome=2; prev_end_zone=75; same_prev_player=1
Given the context, the next event code is:<|im_end|>
<|im_start|>assistant
<think>

</think>




In [ ]:
# ============================================================
# Run 3 v3 inference: v1 zero-shot prompt + RANDOM sampling (n_shot=0).
# Re-uses run_batch (fp16 + efficient SDPA kernel, from the timing cell).
# n_shot=0 -> prompts are ~2k tokens again, so batch 8 fits fine (v1 speed).
# Saves to run3_v3_probs.npz so v1/v2 results are not overwritten.
# ============================================================
import gc
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

v3 timing: 3.92 s/batch of 8 | n_samples=2448 (30.4%) | est 20.0 min
  800/2448 | 7.5 min elapsed | 15.4 min remaining
  1600/2448 | 15.0 min elapsed | 8.0 min remaining
  2400/2448 | 22.5 min elapsed | 0.5 min remaining
  2448/2448 | 23.0 min elapsed | 0.0 min remaining

v3 inference done: 2448 windows in 23.0 min

=== v3 A_full_vocab ===
NLL 21.2559 | Acc 0.2880 | Macro-F1 0.0298 | ECE 0.2880

=== v3 B_renormalised ===
NLL 3.3891 | Acc 0.2880 | Macro-F1 0.0298 | ECE 0.5720

=== v3 B: per-class report ===
                precision    recall  f1-score   support

    PASS_SHORT      0.288     1.000     0.447       705
     PASS_LONG      0.000     0.000     0.000       525
         CARRY      0.000     0.000     0.000       524
         CROSS      0.000     0.000     0.000        50
     CLEARANCE      0.000     0.000     0.000        60
      RECOVERY      0.000     0.000     0.000       163
          DUEL      0.000     0.000     0.000        88
  INTERCEPTION      0.000     0.000    

NameError: name 'Counter' is not defined

In [2]:
# v3 prediction distribution, recomputed CPU-only from the saved artefact.
# (The run cell above printed all metrics, then hit a NameError at this print;
# the saved npz itself is complete, so this reproduces the missing lines.)
# Self-contained: needs only numpy + the Drive artefact, no model or GPU.
import os
try:
    assert os.path.isdir("/content/drive/MyDrive/soccer_mi")
except AssertionError:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np
from collections import Counter

EVENTS = ["PASS_SHORT", "PASS_LONG", "CARRY", "CROSS", "CLEARANCE", "RECOVERY",
          "DUEL", "INTERCEPTION", "SHOT", "GK_ACTION", "DRIBBLE", "LOSS",
          "FOUL_COMMITTED", "FOUL_WON", "OTHER"]   # class index = position here

d = np.load("/content/drive/MyDrive/soccer_mi/run_identity_context.npz")
gold_idx = d["gold_idx"]
pred_b = d["probs_b"].argmax(axis=1)
print(f"v3 prediction distribution (n={len(gold_idx)}):")
for idx, cnt in Counter(pred_b.tolist()).most_common():
    print(f"  {EVENTS[idx]:<15} {cnt:>4} ({cnt/len(pred_b)*100:.1f}%)")
maj = Counter(gold_idx.tolist()).most_common(1)[0]
print(f"majority class: {EVENTS[maj[0]]} = {maj[1]/len(gold_idx)*100:.1f}% of sampled windows")

v3 prediction distribution (n=2448):
  PASS_SHORT      2448 (100.0%)
majority class: PASS_SHORT = 28.8% of sampled windows


In [ ]:
# persist the completed identity+context baseline (v3 run above) to Drive
import shutil, os
dst = "/content/drive/MyDrive/soccer_mi/run_identity_context.npz"
shutil.copy("/content/run3_v3_probs.npz", dst)
print("saved:", dst, f"({os.path.getsize(dst)/1e6:.1f} MB)")

saved: /content/drive/MyDrive/soccer_mi/run_identity_context.npz (0.3 MB)


## Baselines: identity-only and context-only (full test set)

The same zero-shot prompt, context window (16 past events, stride 8), and A/B read-outs as the identity+context run above — only the features rendered per past event change:

- **identity** — the 4 identity features: `player, team, role, competition`
- **context** — the 14 context features: `zone, x, y, period, minute_bin, dt, t_poss, score_diff, poss_len, possession_flag, prev_action, prev_outcome, prev_end_zone, same_prev_player`

Both run over the **full test set (8,057 windows)** — no time budget. Each saves `run_<name>.npz` (probs A/B + gold) directly to `/content/drive/MyDrive/soccer_mi/`. The pipeline cell below defines everything once; the two run cells are one-liners, so adding more feature subsets later is just another `FEATURE_SETS` entry.

In [ ]:
# ============================================================
# Shared pipeline for the feature-set baselines (identity / context).
# Same zero-shot prompt as the identity+context run above, same 16-event
# context, same A/B read-outs. Only the features rendered per past event
# change. Runs over the FULL test set (no time budget) and saves
# probs + gold to /content/drive/MyDrive/soccer_mi/run_<name>.npz
# ============================================================
import gc, math, random, time
from collections import Counter
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# Run: identity-only (4 features), full test set, saves run_identity.npz
res_identity = run_feature_set("identity", FEATURE_SETS["identity"])

=== run 'identity': 4 features ['player', 'team', 'role', 'competition'] ===


identity prompts:   0%|          | 0/8057 [00:00<?, ?it/s]

rendered 8057 prompts; example tail:
ht Center Back; competition=La Liga
14. player=Luis Alberto Suárez Díaz; team=Barcelona; role=Center Forward; competition=La Liga
15. player=Simon Thorup Kjær; team=Sevilla; role=Right Center Back; competition=La Liga
16. player=Jesús Navas González; team=Sevilla; role=Right Midfield; competition=La Liga
Given the context, the next event code is:<|im_end|>
<|im_start|>assistant
<think>

</think>


OK: 100/100 sampled prompts keep the label as 1 token
timing: 1.19 s/batch of 8 | est 19.9 min

  800/8057 | 2.0 min elapsed | 18.4 min remaining
  1600/8057 | 4.0 min elapsed | 16.0 min remaining
  2400/8057 | 5.9 min elapsed | 13.9 min remaining
  3200/8057 | 7.8 min elapsed | 11.9 min remaining
  4000/8057 | 9.8 min elapsed | 9.9 min remaining
  4800/8057 | 11.6 min elapsed | 7.9 min remaining
  5600/8057 | 13.5 min elapsed | 5.9 min remaining
  6400/8057 | 15.3 min elapsed | 4.0 min remaining
  7200/8057 | 17.2 min elapsed | 2.1 min remaining
  8000/80

In [ ]:
# Run: context-only (14 features), full test set, saves run_context.npz
res_context = run_feature_set("context", FEATURE_SETS["context"])

=== run 'context': 14 features ['zone', 'x', 'y', 'period', 'minute_bin', 'dt', 't_poss', 'score_diff', 'poss_len', 'possession_flag', 'prev_action', 'prev_outcome', 'prev_end_zone', 'same_prev_player'] ===


context prompts:   0%|          | 0/8057 [00:00<?, ?it/s]

rendered 8057 prompts; example tail:
ff=3; poss_len=13; possession_flag=1; prev_action=LOSS; prev_outcome=1; prev_end_zone=60; same_prev_player=0
16. zone=45; x=0.433; y=0.700; period=1; minute_bin=0; dt=0.9; t_poss=0.9; score_diff=3; poss_len=0; possession_flag=0; prev_action=PASS_SHORT; prev_outcome=0; prev_end_zone=45; same_prev_player=0
Given the context, the next event code is:<|im_end|>
<|im_start|>assistant
<think>

</think>


OK: 100/100 sampled prompts keep the label as 1 token
timing: 3.19 s/batch of 8 | est 53.6 min

  800/8057 | 5.5 min elapsed | 50.2 min remaining
  1600/8057 | 11.1 min elapsed | 44.6 min remaining
  2400/8057 | 16.6 min elapsed | 39.1 min remaining
  3200/8057 | 22.1 min elapsed | 33.5 min remaining
  4000/8057 | 27.6 min elapsed | 28.0 min remaining
  4800/8057 | 33.1 min elapsed | 22.5 min remaining
  5600/8057 | 38.6 min elapsed | 17.0 min remaining
  6400/8057 | 44.2 min elapsed | 11.4 min remaining
  7200/8057 | 49.7 min elapsed | 5.9 min remaining
 

## Audit: is the inference loop sound?

Two diagnostic cells (cheap — a handful of forward passes, no full runs). They check: (A) artefact integrity and gold-class balance on disk; (B) label/vocabulary assumptions against the parquet; (C) the model's true full-vocabulary top-10 first tokens per feature set; (D) batch-order invariance, i.e. the same prompt in different batch slots yields identical logits (rules out cross-batch state or history effects); (E) input sensitivity, i.e. altered contexts change the logits (the model does read the context). The second cell ranks all 15 letters in the full vocabulary and shows a free generation sample.

Headline finding: the model never literally emits a letter — its top first-tokens are analysis words (`Based`, `Given`, …; `a` ranks ~40th). The "100% PASS_SHORT" result means: conditioned on emitting a letter code, `a` always has the highest logit.

In [ ]:
# ============================================================
# AUDIT: is the inference loop sound, and why 100% PASS_SHORT?
#  A. npz integrity + gold-class distribution on disk
#  B. label/vocab assumptions (parquet action names == EVENTS?)
#  C. full-vocab top-10 FIRST token per feature set (is 'a' really
#     the model's top choice, or only top among the 15 letters?)
#  D. batch-order invariance (same prompt in different batch position
#     -> identical logits; proves no history/state leaks across batches)
#  E. input sensitivity (shuffled context -> logits change;
#     proves the model actually reads the context)
# ============================================================
import numpy as np

print("--- A. npz artefacts on Drive ---")
for f in ["run_identity_context", "run_identity", "run_context"]:
    d = np.load(f"/content/drive/MyDrive/soccer_mi/{f}.npz")
    bc = np.bincount(d["gold_idx"])
    print(f"{f}: { {k: d[k].shape for k in d.files} } "
          f"gold_majority={EVENTS[int(bc.argmax())]} {bc.max()/len(d['gold_idx'])*100:.1f}%")

print("\n--- B. assumptions ---")
print("parquet action names == EVENTS:", set(df["action"].dropna().unique()) == set(EVENTS))
print("action_id -> action:", dict(sorted(action_map.items())))
print("LABEL_IDS a..o:", [LABEL_IDS[l] for l in LABELS])
print("tokenizer padding:", tok.padding_side, "| pad_token:", repr(tok.pad_token))

def first_logits(prompt):
    enc = tok([prompt], return_tensors="pt", padding=True,
              padding_side="left", add_special_tokens=False).to(model.device)
    assert "attention_mask" in enc, "no attention mask!"
    with torch.no_grad(), sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):
        out = model(**enc, logits_to_keep=1, use_cache=False)
    return out.logits[0, 0].float(), enc

def make_prompt(keys, ctx_rows):
    r = make_render_event(keys)
    ctx = "\n".join(f"{k+1}. {r(row)}" for k, row in enumerate(ctx_rows))
    legend = ", ".join(f"{l}={e}" for l, e in LABEL2EVENT.items())
    user = ("You are predicting the next event in a soccer match.\n"
            f"Event codes: {legend}\n"
            f"Context (past 16 events, oldest -> newest):\n{ctx}\n"
            "Given the context, the next event code is:")
    return tok.apply_chat_template([{"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)

ctx0 = windows[0][0]
prompts_by_set = {name: make_prompt(keys, ctx0) for name, keys in FEATURE_SETS.items()}

print("\n--- C. full-vocab top-10 next tokens (one window, per feature set) ---")
for name, p in prompts_by_set.items():
    logits, _ = first_logits(p)
    vals, ids = logits.topk(10)
    print(f"[{name}]")
    for v, i in zip(vals.tolist(), ids.tolist()):
        mark = " <== label 'a'=PASS_SHORT" if i == LABEL_IDS["a"] else ""
        print(f"  {tok.decode([i])!r} (id {i}): {v:.2f}{mark}")

print("\n--- D. batch-order invariance (full 18-feature prompt) ---")
pa, pb = prompts_by_set["identity+context"], make_prompt(FEATURE_SETS["identity+context"], windows[1][0])
la = run_batch([pa, pb])          # pa at position 0
lb = run_batch([pb, pa])          # pa at position 1 (different padding!)
d_pa = (la[0] - lb[1]).abs().max().item()
d_pb = (la[1] - lb[0]).abs().max().item()
print(f"max|logit| diff for same prompt in different batch slot: pa={d_pa:.2e}, pb={d_pb:.2e}")
print("=> batches are independent (no cross-batch state)" if max(d_pa, d_pb) < 1e-3 else "=> PROBLEM: batch position changes logits!")

print("\n--- E. input sensitivity (does the context matter at all?) ---")
l_orig, _ = first_logits(pa)
shuffled = list(ctx0); shuffled = shuffled[::-1]   # reverse event order
l_shuf, _ = first_logits(make_prompt(FEATURE_SETS["identity+context"], shuffled))
l_win1, _ = first_logits(pb)
print(f"L2(orig || reversed-context): {(l_orig - l_shuf).norm().item():.1f}")
print(f"L2(orig || different-window): {(l_orig - l_win1).norm().item():.1f}")
print(f"L2 scale of logits: {l_orig.norm().item():.1f}")
a_id = LABEL_IDS["a"]
for tag, l in [("orig", l_orig), ("reversed", l_shuf), ("other-window", l_win1)]:
    probs15 = torch.softmax(l[label_id_tensor], dim=-1)
    print(f"  P(a=PASS_SHORT | 15-way) [{tag}]: {probs15[0].item():.3f} | argmax letter: {LABELS[int(probs15.argmax())]}")

--- A. npz artefacts on Drive ---
run_identity_context: {'probs_a': (2448, 15), 'probs_b': (2448, 15), 'gold_idx': (2448,), 'n_samples': ()} gold_majority=PASS_SHORT 28.8%
run_identity: {'probs_a': (8057, 15), 'probs_b': (8057, 15), 'gold_idx': (8057,), 'n_samples': ()} gold_majority=PASS_SHORT 29.6%
run_context: {'probs_a': (8057, 15), 'probs_b': (8057, 15), 'gold_idx': (8057,), 'n_samples': ()} gold_majority=PASS_SHORT 29.6%

--- B. assumptions ---
parquet action names == EVENTS: True
action_id -> action: {0: 'PASS_SHORT', 1: 'PASS_LONG', 2: 'CROSS', 3: 'CARRY', 4: 'DRIBBLE', 5: 'SHOT', 6: 'CLEARANCE', 7: 'RECOVERY', 8: 'INTERCEPTION', 9: 'DUEL', 10: 'FOUL_WON', 11: 'FOUL_COMMITTED', 12: 'GK_ACTION', 13: 'LOSS', 14: 'OTHER'}
LABEL_IDS a..o: [64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78]
tokenizer padding: right | pad_token: '<|endoftext|>'

--- C. full-vocab top-10 next tokens (one window, per feature set) ---
[identity+context]
  'Based' (id 28715): 35.88
  'Given' (id

In [ ]:
# AUDIT-2: where do the 15 letters rank in the FULL vocab,
# and what would the model actually emit as its first tokens?
p = prompts_by_set["identity+context"]
logits, _ = first_logits(p)
order = logits.argsort(descending=True)
ranks = {f"{l}={LABEL2EVENT[l]}": int((order == LABEL_IDS[l]).nonzero().item()) + 1 for l in LABELS}
print("full-vocab rank of each letter (1 = model's actual top choice):")
for k, v in sorted(ranks.items(), key=lambda kv: kv[1]):
    print(f"  {k:<22} rank {v}")

enc = tok([p], return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**enc, max_new_tokens=25, do_sample=False)
print("\nactual 25-token generation:")
print(repr(tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)))

full-vocab rank of each letter (1 = model's actual top choice):
  a=PASS_SHORT           rank 40
  g=DUEL                 rank 110
  b=PASS_LONG            rank 134
  c=CARRY                rank 159
  d=CROSS                rank 190
  i=SHOT                 rank 204
  l=LOSS                 rank 210
  e=CLEARANCE            rank 370
  f=RECOVERY             rank 410
  j=GK_ACTION            rank 510
  n=FOUL_WON             rank 641
  h=INTERCEPTION         rank 815
  o=OTHER                rank 1085
  m=FOUL_COMMITTED       rank 1365
  k=DRIBBLE              rank 1785

actual 25-token generation:
'Based on the context provided, the next event is likely to be a **CROSS** or **SHOT**. \n\nLooking'


## Artefacts and results

Each run saves `probs_a` (full-vocab read-out), `probs_b` (15-way renormalised read-out) and `gold_idx` (integer class labels) as an `.npz` file on Drive. The cell below lists the artefacts; the results table follows.

In [ ]:
# verify all three baseline artefacts are on Drive
import os
for f in sorted(os.listdir("/content/drive/MyDrive/soccer_mi")):
    if f.endswith(".npz"):
        print(f, f"{os.path.getsize('/content/drive/MyDrive/soccer_mi/'+f)/1e6:.2f} MB")

run_context.npz 1.03 MB
run_identity.npz 1.03 MB
run_identity_context.npz 0.31 MB


## Results summary

Zero-shot Qwen3-1.7B, greedy, first-token logits, B = 15-way renormalised read-out (headline).

| Run | Features | N | B: NLL | B: Acc | B: Macro-F1 | B: ECE | A: NLL | Argmax letter |
|---|---|---|---|---|---|---|---|---|
| identity+context | 18 | 2448 (30% random) | 3.39 | 0.288 | 0.030 | 0.572 | 21.26 | always `a` (=PASS_SHORT) |
| identity | 4 | 8057 (full) | 3.25 | 0.296 | 0.030 | 0.555 | 23.85 | always `a` |
| context | 14 | 8057 (full) | 4.67 | 0.296 | 0.030 | 0.618 | 26.25 | always `a` |

Accuracy == majority rate in all three runs (28.8% / 29.6% / 29.6%): a degenerate constant predictor under the 15-way read-out.

**Audit caveat (see the two AUDIT cells above):** the model does *not* actually emit `a`. Its true top-1 next tokens are analysis words (`Based`, `Given`, `The`, … — `a` ranks only ~40th in the full vocab), and free generation starts with e.g. *"Based on the context provided, the next event is likely to be a CROSS or SHOT…"*. So the correct reading is "conditioned on emitting a letter code, `a` always wins" — the prompt fails to constrain the model to the letter format. Metrics remain well-defined measurements of the letter logits; batch-order invariance (0.00 diff) and input-sensitivity checks confirm the inference loop itself is sound (no state leakage, no target leakage, model does read the context).

Artefacts: `run_identity_context.npz`, `run_identity.npz`, `run_context.npz` in `/content/drive/MyDrive/soccer_mi/`.